# Transfer Learning — CrVO₃ Self-Energy Prediction

Demonstrates how the SVO-pretrained LSTM autoencoder can be fine-tuned with just
**two nwarmup conditions** of CrVO₃ (CVO) data and then generalise to the remaining
held-out conditions.

**Two-phase fine-tuning strategy:**
- **Phase 1 (30 epochs):** Freeze the encoder layers; adapt only the decoder to the
  new material's distribution.  
- **Phase 2 (40 epochs):** Unfreeze all layers and continue with a low learning rate
  (1 × 10⁻⁵) to avoid catastrophic forgetting.

**Material:** CrVO₃ (CVO) — different d-electron count and bandwidth from SVO  
**Pretrained model:** `models/trained_model_svo.keras`  
**Fine-tuning data:** nwarmup ∈ {100 000, 250 000}  
**Test data:** nwarmup ∈ {150 000, 200 000, 300 000, 350 000}

## 0. Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Lambda, Embedding, BatchNormalization,LSTM, RepeatVector, TimeDistributed, concatenate, Reshape
from tensorflow.keras.losses import mse, binary_crossentropy
from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model
from keras.optimizers import Adam

## 1. Load CVO Data and Define Train / Test Split

We use only 2 of 6 available nwarmup conditions for fine-tuning.
The label normalisation is kept on the *SVO* scale so the pretrained
model's internal representations remain meaningful.

In [ ]:
#### read data files CVO


import numpy as np
data_CVO = np.load('data/dataCVO.npz')
nwarmup_all = data_CVO['nwarupsteps']
print(np.unique(nwarmup_all))

real_sigma_all = data_CVO['realpart_sigma']
img_sigma_all = data_CVO['imgpart_sigma']

real_g0_all = data_CVO['realpart_g0']
img_g0_all = data_CVO['imgpart_g0']

mask_train = np.isin(nwarmup_all,[100000 ,250000] )
mask_test = ~np.isin(nwarmup_all,[100000,250000] )





## 2. Normalise Labels (SVO-compatible scale)

Critically, we re-use the SVO min/max values (50 000 – 580 000) for
normalisation rather than CVO's own range. This ensures the label embedding
aligns with what the pretrained encoder already learned.

In [ ]:
#preprocess
min = 50000
max = 580000 # these are hardcoded from the SVO nwarmupset
min_max_normal = lambda x: (x-min/(max - min))
all_labels = min_max_normal(nwarmup_all) #nwarmup
X_all = img_g0_all #min_max_normal(img_train)
Y_all = img_sigma_all


## 3. Load Pretrained SVO Model and Freeze Encoder

Load `models/trained_model_svo.keras` and freeze the first 4 layers
(input + label input + RepeatVector + concatenate) to preserve the
encoder's learned G₀ representation.

In [ ]:
trained_model =load_model('models/trained_model_svo.keras')


# Freeze some layers to preserve pre-trained knowledge
for layer in trained_model.layers[:4]:  # Example: freezing the entire encoding layer
    layer.trainable = False
trained_model.compile(optimizer='adam', loss='mse')


## 4. Phase 1 — Fine-tune with Frozen Encoder (30 epochs)

Only the decoder weights update. The encoder is protected from the
small CVO dataset to prevent overfitting.

In [ ]:
trained_model.fit([X_all[mask_train],all_labels[mask_train]],Y_all[mask_train].reshape(-1,Y_all.shape[1] , 1),epochs=30, batch_size=32,validation_split=0.2)

## 5. Phase 2 — Full Fine-tune at Low Learning Rate (40 epochs)

Unfreeze all layers and continue training at LR = 1 × 10⁻⁵.
This allows the encoder to make small adjustments for CVO while
the low LR prevents the decoder from drifting.

In [ ]:


#### fine-tune by unfreezing all layers

for layer in trained_model.layers:  # Example: freezing the first two layers
    layer.trainable = True
trained_model.compile(optimizer=Adam(1e-5), loss='mse')
trained_model.fit([X_all[mask_train],all_labels[mask_train]],Y_all[mask_train].reshape(-1,Y_all.shape[1] , 1),epochs=40, batch_size=32,validation_split=0.2)



## 6. Predict on Held-out CVO Samples

In [ ]:
Y_pred = trained_model.predict([X_all[mask_test], all_labels[mask_test]])

## 7. Evaluate — Plot Predicted vs DMFT Profiles

Each panel shows the mean predicted Im Σ(iω) compared to the DMFT
truth, averaged over all spin-orbital channels at that nwarmup value.
Plots are saved to `plots/`.

In [ ]:
import os
os.makedirs('plots', exist_ok=True)
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica'],
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 10,
    'axes.linewidth': 2.0,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.minor.width': 0.6,
    'ytick.minor.width': 0.6,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'axes.labelpad': 10
})


y_test = Y_all[mask_test]

nwarmup_labels = nwarmup_all[mask_test]
unique_labels = np.unique(nwarmup_labels)


rms_all=[]

rms= lambda x: np.sqrt(np.mean(x*x))

### frequency
freq=np.linspace(-161,161,2050)
mask = (freq > 0) & (freq < 10)
freq_sel = freq[mask]



'''
for i in range(20):
    error = rms(Y_pred[i] - Y_all[mask_test][i])
    print('nwarmup step, error_rms', nwarmup_all[mask_test][i], error)
    plt.plot(Y_pred[i],'b.-',alpha=0.5,label='prediction')
    plt.plot(Y_all[mask_test][i],'k-',label='MC truth')
    plt.ylabel('Sigma_iw', fontsize=20)
    plt.title(f'Nwarmup step:{nwarmup_all[mask_test][i]}',fontsize=20)
    plt.tight_layout()
    plt.legend()
   # plt.savefig(f'sno_plots/g02iw_{nwarmup_labels[i]}.png')
    plt.show()

'''
#### make avg profile per iteration here

#### plot truth vs prediction avg plot for each unique warmup step
for i in np.arange(len(unique_labels)):
    y_pred_mean = np.mean(Y_pred[nwarmup_labels==unique_labels[i]],axis=0)
    y_truth_mean =  np.mean(y_test[nwarmup_labels==unique_labels[i]],axis=0)
    rms_all.append(rms((y_pred_mean.flatten()-y_truth_mean.flatten())/y_truth_mean.flatten()))
    print(unique_labels[i],y_pred_mean.flatten())
    #rms_all.append(rms((y_pred_mean-y_truth_mean)/y_truth_mean))
    y_pred_std = np.std(Y_pred[nwarmup_labels==unique_labels[i]],axis=0)
    plt.plot(freq_sel,y_pred_mean.flatten(), marker='s',color='green',markersize=6.5,label='ML')
    #plt.plot(freq_sel,y_pred_mean,'b.-',label='DMFT')
    plt.plot(freq_sel,y_truth_mean,'k-',lw=3,label='DMFT',zorder=3)
    plt.ylabel(r' Mean Im $\Sigma_{i\omega}$ (1/eV)', fontsize=20)
    plt.xlabel('$\omega$ (eV)',fontsize=20)
    
    plt.legend(fontsize=15)
    plt.tight_layout()
    plt.savefig(f'plots/TL_CVO_prediction_new_{unique_labels[i]}.eps',format='eps',dpi=600,bbox_inches='tight')
    plt.show()
#np.save('TL_CVO_new2.npy',np.array([unique_labels,rms_all]))

In [ ]:
print(rms_all)